In [1]:
pip install arch

   ---------------------------------------- 0.0/929.7 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/929.7 kB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 929.7/929.7 kB 3.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
import scipy.stats as stats
from arch import arch_model
import time

# ── Fetch BTC 1h bars from Binance (no API key needed) ──────────────────────
def get_btc_hourly(n_bars=900):
    url = "https://data-api.binance.vision/api/v3/klines"
    all_bars = []
    end_time = None
    
    while len(all_bars) < n_bars:
        params = {"symbol": "BTCUSDT", "interval": "1h", "limit": 1000}
        if end_time:
            params["endTime"] = end_time
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        bars = r.json()
        if not bars:
            break
        all_bars = bars + all_bars
        end_time = bars[0][0] - 1
        if len(bars) < 1000 or len(all_bars) >= n_bars:
            break
        time.sleep(0.1)
    
    all_bars = all_bars[-n_bars:]
    df = pd.DataFrame(all_bars, columns=[
        'open_time','open','high','low','close','volume',
        'close_time','quote_vol','trades','taker_base','taker_quote','ignore'
    ])
    df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')
    df.set_index('close_time', inplace=True)
    prices = df['close'].astype(float)
    prices.index.name = 'date'
    return prices

# Fetch data
print("Fetching BTC hourly data from Binance...")
prices = get_btc_hourly(n_bars=900)
print(f"Got {len(prices)} bars.")
print(f"Latest bar: {prices.index[-1]}")
print(f"Latest BTC price: ${float(prices.iloc[-1]):,.2f}")

# Model setup
np.random.seed(42)
log_ret   = np.log(prices / prices.shift(1)).dropna()
mu_hourly = log_ret.mean()
S0        = prices.iloc[-1]
dt        = 1
n_sims    = 10_000

am  = arch_model(log_ret * 100, vol='Garch', p=1, q=1, dist='studentst')
res = am.fit(disp='off')
sigma_garch = res.conditional_volatility / 100
resid       = (log_ret * 100 - res.params['mu']) / res.conditional_volatility
nu          = max(4, stats.t.fit(resid, floc=0, fscale=1)[0])

print(f"\nModel fitted. nu = {nu:.2f}")
print(f"Latest hourly sigma = {sigma_garch.iloc[-1]*100:.4f}%")

def rolling_entropy(x, window=60, bins=20):
    def ent(v):
        p, _ = np.histogram(v, bins=bins, density=True)
        p = p[p > 0]
        return -np.sum(p * np.log(p))
    return x.rolling(window).apply(ent, raw=True)

H_series   = rolling_entropy(resid)
M_series   = log_ret.abs().rolling(60).mean()
bar_sigma2 = (sigma_garch**2).mean()
redundancy  = (1 + 0.1 * np.log1p(
    prices.rolling(5).var() / prices.rolling(20).var()
)).fillna(1)
info_filter = (H_series > H_series.mean()).fillna(False).astype(float)

H_max, M_max = H_series.max(), M_series.max()
α0, δ0 = 0.5, 0.3
if H_max > 0 and M_max > 0 and α0 * H_max + δ0 * M_max >= 1:
    fac = 0.95 / (α0 * H_max + δ0 * M_max)
    α0 *= fac; δ0 *= fac

base_params = {'alpha': α0, 'delta': δ0, 'gamma': 0.2, 'kappa': 0.1, 'eta': 1e-3}

def update_params(p, sigma2, bar_sigma2, t):
    err = sigma2 - bar_sigma2
    lr  = p['eta'] / (1 + t**0.55)
    p['gamma'] = np.clip(p['gamma'] + lr * err, 0.01, 0.5)
    return p

def simulate_cyber_gbm(S0, mu, sigma_series, H, M,
                       params, bar_sigma2, n_steps, dt=1, eps=1e-6):
    S = np.zeros(n_steps + 1)
    S[0] = float(S0)
    sigma2 = float(sigma_series.iloc[-1]) ** 2
    _H_max = float(H.max()) if H.max() > 0 else 1.0
    _M_max = float(M.max()) if M.max() > 0 else 1.0
    _red   = float(redundancy.iloc[-1]) if not np.isnan(redundancy.iloc[-1]) else 1.0
    _inf   = float(info_filter.iloc[-1])

    for t in range(1, n_steps + 1):
        H_val   = min(float(H.iloc[-1]) / _H_max, 1.0) if not np.isnan(H.iloc[-1]) else 0.0
        M_val   = min(float(M.iloc[-1]) / _M_max, 1.0) if not np.isnan(M.iloc[-1]) else 0.0
        crisis  = (H_val > 0.8) or (M_val > 0.8)
        delta_t = params['delta'] if crisis else 0.0
        sigma2  = (
            float(sigma_series.iloc[-1])**2 * (1 + params['alpha']*H_val + delta_t*M_val)
            + params['gamma'] * (bar_sigma2 - sigma2)
        )
        sigma2 *= max(1e-12, _red)
        sigma2 *= 1 + 0.5 * _inf
        sigma2  = max(eps, min(sigma2, 0.5))
        Z       = np.random.standard_t(nu) * np.sqrt((nu - 2) / nu)
        S[t]    = S[t-1] * np.exp((mu - 0.5*sigma2)*dt + np.sqrt(sigma2*dt)*Z)
        params  = update_params(params, sigma2, bar_sigma2, t)
    return S

def simulate_mc(S0, mu, sigma_series, H, M, bar_sigma2, n_sims=10_000, n_steps=1):
    out = np.zeros((n_sims, n_steps + 1))
    for i in range(n_sims):
        out[i] = simulate_cyber_gbm(
            S0, mu, sigma_series, H, M,
            base_params.copy(), bar_sigma2, n_steps, dt
        )
    return out

# Quick test
print("\nRunning quick test (1000 simulations)...")
paths  = simulate_mc(S0, mu_hourly, sigma_garch, H_series, M_series,
                     bar_sigma2, n_sims=1000, n_steps=1)
S_t1   = paths[:, 1]
low95, high95 = np.percentile(S_t1, [2.5, 97.5])

print(f"\n✅ CELL 2 COMPLETE")
print(f"   Current BTC : ${float(S0):,.2f}")
print(f"   95% CI next hour: ${low95:,.2f} — ${high95:,.2f}")
print(f"   Width       : ${high95 - low95:,.2f}")

Fetching BTC hourly data from Binance...
Got 900 bars.
Latest bar: 2026-05-03 06:59:59.999000
Latest BTC price: $78,150.85

Model fitted. nu = 11.67
Latest hourly sigma = 0.2583%

Running quick test (1000 simulations)...

✅ CELL 2 COMPLETE
   Current BTC : $78,150.85
   95% CI next hour: $77,570.54 — $78,785.74
   Width       : $1,215.21


In [4]:
import json
from tqdm import tqdm

def run_backtest(prices, train_window=180, n_test=720, n_sims=3000):
    log_ret_full = np.log(prices / prices.shift(1)).dropna()
    prices_clean = prices.iloc[1:]  # align with log_ret

    results = []
    start_idx = train_window
    end_idx = min(start_idx + n_test, len(prices_clean) - 1)

    print(f"Backtesting {end_idx - start_idx} bars...")

    for i in tqdm(range(start_idx, end_idx)):
        # ── NO PEEKING: only use data[:i] ──
        hist_ret = log_ret_full.iloc[i - train_window:i]
        S0_bt    = float(prices_clean.iloc[i])
        actual   = float(prices_clean.iloc[i + 1])

        try:
            am_bt  = arch_model(hist_ret * 100, vol='Garch', p=1, q=1, dist='studentst')
            res_bt = am_bt.fit(disp='off', show_warning=False)
            sig_bt = res_bt.conditional_volatility / 100
            resid_bt = (hist_ret * 100 - res_bt.params['mu']) / res_bt.conditional_volatility
            nu_bt  = max(4, stats.t.fit(resid_bt, floc=0, fscale=1)[0])

            H_bt      = rolling_entropy(resid_bt)
            M_bt      = hist_ret.abs().rolling(60).mean()
            bar_s2_bt = (sig_bt**2).mean()

            red_bt  = (1 + 0.1 * np.log1p(
                prices_clean.iloc[i-180:i].rolling(5).var() /
                prices_clean.iloc[i-180:i].rolling(20).var()
            )).fillna(1)
            inf_bt  = (H_bt > H_bt.mean()).fillna(False).astype(float)

            H_max_bt = H_bt.max() if H_bt.max() > 0 else 1.0
            M_max_bt = M_bt.max() if M_bt.max() > 0 else 1.0
            a0, d0 = 0.5, 0.3
            if H_max_bt > 0 and M_max_bt > 0 and a0*H_max_bt + d0*M_max_bt >= 1:
                fac = 0.95 / (a0*H_max_bt + d0*M_max_bt)
                a0 *= fac; d0 *= fac
            bp = {'alpha': a0, 'delta': d0, 'gamma': 0.2, 'kappa': 0.1, 'eta': 1e-3}

            # Simulate n_sims paths
            sims = np.zeros(n_sims)
            s2_last = float(sig_bt.iloc[-1])**2
            H_val = min(float(H_bt.iloc[-1])/H_max_bt, 1.0) if not np.isnan(H_bt.iloc[-1]) else 0.0
            M_val = min(float(M_bt.iloc[-1])/M_max_bt, 1.0) if not np.isnan(M_bt.iloc[-1]) else 0.0
            crisis = (H_val > 0.8) or (M_val > 0.8)
            dt_c = bp['delta'] if crisis else 0.0
            sigma2 = (sig_bt.iloc[-1]**2 * (1 + bp['alpha']*H_val + dt_c*M_val)
                      + bp['gamma']*(bar_s2_bt - s2_last))
            sigma2 *= max(1e-12, float(red_bt.iloc[-1]))
            sigma2 *= 1 + 0.5*float(inf_bt.iloc[-1])
            sigma2  = max(1e-8, min(sigma2, 0.5))
            mu_bt   = float(hist_ret.mean())
            Z = np.random.standard_t(nu_bt, size=n_sims) * np.sqrt((nu_bt-2)/nu_bt)
            sims = S0_bt * np.exp((mu_bt - 0.5*sigma2) + np.sqrt(sigma2)*Z)
            low95, high95 = np.percentile(sims, [2.5, 97.5])

        except Exception as e:
            vol    = hist_ret.std()
            low95  = S0_bt * np.exp(-1.96 * vol)
            high95 = S0_bt * np.exp(+1.96 * vol)

        width = high95 - low95
        hit   = int(low95 <= actual <= high95)
        alpha = 0.05
        if actual < low95:
            winkler = width + (2/alpha)*(low95 - actual)
        elif actual > high95:
            winkler = width + (2/alpha)*(actual - high95)
        else:
            winkler = width

        results.append({
            "timestamp": str(prices_clean.index[i]),
            "S0": S0_bt,
            "low_95": low95,
            "high_95": high95,
            "actual": actual,
            "hit": hit,
            "width": width,
            "winkler": winkler
        })

    return results

# ── RUN ──────────────────────────────────────────────────────────────────────
print("Starting Part A backtest (~15-25 mins)...")
backtest_results = run_backtest(prices, train_window=180, n_test=720, n_sims=3000)

# Save
with open("backtest_results.jsonl", "w") as f:
    for r in backtest_results:
        f.write(json.dumps(r) + "\n")

# Metrics
hits     = [r['hit']    for r in backtest_results]
widths   = [r['width']  for r in backtest_results]
winklers = [r['winkler']for r in backtest_results]

coverage     = sum(hits) / len(hits)
avg_width    = sum(widths) / len(widths)
mean_winkler = sum(winklers) / len(winklers)

print(f"\n{'='*40}")
print(f"Part A Results ({len(backtest_results)} predictions)")
print(f"{'='*40}")
print(f"Coverage (95%):  {coverage:.4f}  ← target ~0.95")
print(f"Avg Width:       ${avg_width:,.2f}")
print(f"Mean Winkler:    ${mean_winkler:,.2f}")
print(f"\n✅ Saved to backtest_results.jsonl")
print(f"   WRITE DOWN: coverage={coverage:.4f}, winkler={mean_winkler:.2f}")

Starting Part A backtest (~15-25 mins)...
Backtesting 718 bars...


 88%|████████▊ | 630/718 [00:36<00:05, 16.11it/s]c:\Users\USER\anaconda3\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.09132. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
 88%|████████▊ | 632/718 [00:36<00:05, 16.07it/s]c:\Users\USER\anaconda3\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.09516. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=


Part A Results (718 predictions)
Coverage (95%):  0.9680  ← target ~0.95
Avg Width:       $1,456.42
Mean Winkler:    $1,783.19

✅ Saved to backtest_results.jsonl
   WRITE DOWN: coverage=0.9680, winkler=1783.19


In [5]:
import os
print(os.getcwd())

c:\Users\USER\Desktop\AlphaAI
